This helper makes sure that the packages that rebuilt successfully with Python 3.14 but fail to install due to their dependencies not yet rebuilt have bugzillas blocked by the appropriate bugzillas of said dependencies.

 1. For every package that has been successfully rebuilt, we see if it has a bugzilla that blocks `PYTHON3.14` and `F43FailsToInstall`.
 2. If so, we parse the initial comment for missing `python3.14dist()` dependencies.
 3. We convert the dependencies to `python3.13dist()` dependencies and find their component.
 4. If the component has a `PYTHON3.14` bugzilla, we mark it as blocking the bugzilla from (1).

In [1]:
from IPython.display import display, Markdown

In [2]:
import bugzilla
bzapi = bugzilla.Bugzilla('bugzilla.redhat.com')
query = bzapi.build_query(product='Fedora')
query['blocks'] = 2322407  # PYTHON3.14
query['limit'] = 20 # Bugzilla page size
query['offset'] = 0
query['status'] = '__open__'
bugz = []
while len(partial := bzapi.query(query)) == 20:
    bugz += partial
    query['offset'] += 20
    print(len(bugz))
bugz += partial

20
40
60
80
100
120
140
160
180
200
220
240
260
280
300
320
340
360
380
400
420
440
460
480
500
520
540
560
580
600
620
640
660


In [3]:
F43FailsToInstall = 2339435

In [4]:
python314 = !cat python314.pkgs

In [5]:
len(pure_fti_bugz := [b for b in bugz if b.component in python314 and F43FailsToInstall in b.blocks])
pure_fti_bugz

[<Bug #2371681 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f51514dff50>,
 <Bug #2371695 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f51514dca60>,
 <Bug #2371696 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f51514dcad0>,
 <Bug #2371732 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f5151298fa0>,
 <Bug #2371736 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f5151299240>,
 <Bug #2371744 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f51512994e0>,
 <Bug #2371751 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f51512996a0>,
 <Bug #2371760 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f51512998d0>,
 <Bug #2371765 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f5151298bb0>,
 <Bug #2371767 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f5151299be0>,
 <Bug #2371773 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f5151299da0>,
 <Bug #2371774 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f5151299e10>,
 <Bug #2371834 on https://bugzilla.redhat.com/xmlrpc.cgi at 0x7f515129add0>,

In [9]:
for bug in pure_fti_bugz:
    display(Markdown(f'# {bug.component}\n[rbz#{bug.id}](https://bugzilla.redhat.com/{bug.id})'))
    comments = bug.getcomments()
    initial_comment = comments[0]
    if initial_comment['creator'] != 'fti-bugs@fedoraproject.org':
        print('Not an automated FTI bug by Miro')
        continue

    blocked_by = set()
    lines = initial_comment['text'].splitlines()
    for line in lines:
        if line.startswith('  - nothing provides '):
            print(line)
            provide = line.split(' ')[5]
            if provide.startswith('('):
                provide = provide[1:]
            if provide.startswith('python3.14dist('):
                old_provide = provide.replace('python3.14dist(', 'python3.13dist(')
                blocker_components = !repoquery -q --repo=rawhide --source --whatprovides "{old_provide}" | pkgname
                if len(set(blocker_components)) == 1:
                    blocker_component = blocker_components[0]
                    blocker_bugz = [b for b in bugz if b.component == blocker_component]
                    if len(blocker_bugz) > 1:
                        print(f"    = too many bugzillas for {blocker_component}")
                        for bug in blocker_bugz:
                            print(f"        https://bugzilla.redhat.com/{bug.id}")
                    elif blocker_bugz:
                        bid = blocker_bugz[0].id
                        print(f"    = blocked by {blocker_component} https://bugzilla.redhat.com/{bid}")
                        blocked_by.add(bid)
                else:
                    print('Check:', blocker_components)
    # print(blocked_by)
    if blocked_by:
       add_blockers = bzapi.build_update(depends_on_add=sorted(blocked_by))
       bzapi.update_bugs([bug.id], add_blockers)

# beets
[rbz#2371681](https://bugzilla.redhat.com/2371681)

  - nothing provides python3.14dist(confuse) >= 1.5 needed by beets-2.0.0-3.fc43.noarch
    = blocked by python-confuse https://bugzilla.redhat.com/2327966


# cobbler
[rbz#2371695](https://bugzilla.redhat.com/2371695)

  - nothing provides python3.14dist(pymongo) needed by cobbler-3.3.7-7.fc43.noarch
    = blocked by python-pymongo https://bugzilla.redhat.com/2356166


# conda
[rbz#2371696](https://bugzilla.redhat.com/2371696)

...

  - nothing provides python3.14dist(pydantic) >= 1.7.4 needed by python3-x3dh-1.0.4-3.fc43.noarch
    = blocked by python-pydantic https://bugzilla.redhat.com/2372054


# rapid-photo-downloader
[rbz#2372211](https://bugzilla.redhat.com/2372211)

  - nothing provides python3.14dist(gphoto2) needed by rapid-photo-downloader-0.9.33-15.fc43.noarch
    = blocked by python-gphoto2 https://bugzilla.redhat.com/2366465


# resalloc-openstack
[rbz#2372214](https://bugzilla.redhat.com/2372214)

  - nothing provides python3.14dist(python-neutronclient) needed by resalloc-openstack-9.8-6.fc43.noarch
    = blocked by python-neutronclient https://bugzilla.redhat.com/2371998
  - nothing provides python3.14dist(python-novaclient) needed by resalloc-openstack-9.8-6.fc43.noarch
    = blocked by python-novaclient https://bugzilla.redhat.com/2372005
